In [1]:
%matplotlib inline
import torch
import random
from d2l import torch as d2l


In [2]:
def synthetic_data(w,b,number_example):
    "y = Xw + b + 噪声 "
    X = torch.normal(0,1,(number_example,len(w)))   #正态分布括号里的两个到底是什么
    y = torch.matmul(X,w)+b
    y = torch.normal(0,0.01,y.shape)
    return X,y.reshape(-1,1)        #这里的因为什么要加reshape 
    



In [3]:
true_w = torch.tensor([2,-3.4])
true_b = 4.2
features, labels = synthetic_data(true_w,true_b,1000)

In [4]:
print("true_w:",true_w.shape) #

true_w: torch.Size([2])


In [5]:
print("features:",features)
print("features.shape",features.shape)

features: tensor([[-0.3266,  1.6755],
        [ 0.2433, -0.5044],
        [ 0.4721,  1.5732],
        ...,
        [ 0.5255,  0.3099],
        [-2.4792,  0.8467],
        [ 1.6521, -0.3032]])
features.shape torch.Size([1000, 2])


In [6]:
print("labels：" ,labels[:10])
print(labels.shape)

labels： tensor([[ 0.0087],
        [ 0.0114],
        [-0.0020],
        [ 0.0025],
        [ 0.0147],
        [ 0.0068],
        [ 0.0022],
        [ 0.0027],
        [-0.0016],
        [ 0.0002]])
torch.Size([1000, 1])


In [7]:
def data_iter(batch_size,features,lables): #我现在要定义一个迭代的数据器
    num_example = len(features)  #获取features的长度生成一个列表
    indices= list(range(num_example))
    random.shuffle(indices)
    for  i in range(0,num_example,batch_size):
        batch_indices = torch.tensor(indices[i:min(i+batch_size,num_example)])  #我从乱的数据里取出前10个
        #print("batch_indices:",batch_indices)
        yield  features[batch_indices],labels[batch_indices]

batch_size = 10 
for X,y in data_iter(batch_size,features,labels):
    print("每批次训练的前十个特征值:",X ,"\n","每批次训练的前十个结果:" ,y)
    break;
    

每批次训练的前十个特征值: tensor([[-1.4851, -0.8158],
        [-0.4115,  0.6870],
        [ 0.5276, -1.0298],
        [-0.3976,  0.3647],
        [-0.3663,  0.0964],
        [ 0.6113,  0.5718],
        [ 1.0270,  0.0400],
        [ 0.4287, -0.7312],
        [-0.6430,  0.4079],
        [ 0.6847, -0.6336]]) 
 每批次训练的前十个结果: tensor([[-0.0247],
        [ 0.0029],
        [ 0.0087],
        [-0.0061],
        [-0.0013],
        [ 0.0055],
        [-0.0131],
        [-0.0139],
        [-0.0118],
        [-0.0031]])


In [8]:
print("labels[10]",labels[760])

labels[10] tensor([0.0059])


In [9]:
def linreg(X,w,b):
    """线性回归 liner regression #matrix multiplaction)"""
    y_hat = torch.matmul(X,w)+b
    return y_hat

In [10]:
def squared_loss(y_hat,y):
    """squared loss"""
    loss_value = (y_hat-y.reshape(y.shape))**2/2    #我这里已经忘记了为什么要在这里reshape 
    return loss_value

In [11]:
def sgd(params,lr,batch_size):
    """stochastic gradient descent"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad/batch_size   #attention here is difference 
            param.grad.zero_()

        

In [12]:
w = torch.normal(0,0.01,size=(2,1),requires_grad=True)
b = torch.zeros(1,requires_grad = True)

In [13]:
lr = 0.03
num_epochs = 3 

for epoch in range(num_epochs):  #这个外层的循环我就没有看懂什么意思
    for x,y in data_iter(batch_size,features,labels):
        l = squared_loss(linreg(x,w,b),y)
        l.sum().backward()
        sgd([w,b], lr,batch_size)
    with torch.no_grad():
            train_l=squared_loss(linreg(features,w,b),labels)

    print(f'epoch{epoch+1},loss{float(train_l.mean()):f}')
    

epoch1,loss0.000053
epoch2,loss0.000052
epoch3,loss0.000052
